# **Install and/or import libraries:**

Install (and import) these libraries on the machine if you already dont have:

**On Kaggle:**

In [2]:
! apt-get install -y libopenmpi-dev
! pip install wheel mpi4py
! pip install gpustat
! pip install noise

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  gfortran gfortran-12 ibverbs-providers javascript-common libcaf-openmpi-3
  libcoarrays-dev libcoarrays-openmpi-dev libfabric1 libgfortran-12-dev
  libhwloc-dev libhwloc-plugins libhwloc15 libibverbs-dev libibverbs1
  libjs-jquery libjs-jquery-ui libmunge2 libnl-3-200 libnl-3-dev
  libnl-route-3-200 libnl-route-3-dev libnuma-dev libopenmpi3 libpmix-dev
  libpmix2 libpsm-infinipath1 libpsm2-2 librdmacm1 libucx0 libxnvctrl0
  openmpi-bin openmpi-common
Suggested packages:
  gfortran-multilib gfortran-doc gfortran-12-multilib gfortran-12-doc apache2
  | lighttpd | httpd libhwloc-contrib-plugins libjs-jquery-ui-docs openmpi-doc
The following NEW packages will be installed:
  gfortran gfortran-12 ibverbs-providers javascript-common libcaf-openmpi-3
  libcoarrays-dev libcoarrays-openmpi-dev libfabric1 libgfortran-12-dev
  libhwloc-dev libhwlo

# **Simulation:**

## NCCL + custom all_to_all version:

In [5]:
%%writefile /kaggle/working/test_mpi.py

import time
import subprocess
import numpy as np
from tqdm import tqdm
from mpi4py import MPI

import cupy as cp
from cupy import cuda
from cupy.cuda import nccl
from cupyx.scipy.fft import fftfreq, fft, ifft, irfft2, rfft2, fftn, irfftn, rfftn


def IC_3D(X, IC_type):
    '''
    This function initializes the velocity field in Fourier space based on the initial condition type
    '''
    print('Inside function')
    if IC_type == 'random_vel':
        # Random velocity initial condition (not a very good IC for 3D turbulence)
        U[0] = cp.random.rand(*X[0].shape)
        U[1] = cp.random.rand(*X[0].shape)
        U[2] = cp.random.rand(*X[0].shape)

        #Resize:
        U[0] /= cp.max(U[0])
        U[1] /= cp.max(U[1])
        U[2] /= cp.max(U[2])

    if IC_type == 'taylor_green':
        # Taylor-Green vortex initial conditions (Check Mortensen (2016) paper)
        U[0] = cp.sin(X[0])*cp.cos(X[1])*cp.cos(X[2])
        U[1] = -cp.cos(X[0])*cp.sin(X[1])*cp.cos(X[2])
        U[2] = 0

    if IC_type == 'taylor_green_noise':
        print('inside taylor_green')
        # Taylor-Green vortex with added noise initial condition
        U[0] = cp.sin(X[0])*cp.cos(X[1])*cp.cos(X[2])
        U[1] = -cp.cos(X[0])*cp.sin(X[1])*cp.cos(X[2])
        U[2] = 0

        #Add white noise:
        epsilon = 0.1
        U[0] += epsilon*cp.random.rand(*U[0].shape)
        U[1] += epsilon*cp.random.rand(*U[1].shape)
        U[2] += epsilon*cp.random.rand(*U[2].shape)

    if IC_type == 'perlin_noise':
        # Perlin noise CURL initial condition

        scale = 1/4 #0.25
        octaves = 5 #2
        persistence = 0.4 #0.5
        lacunarity = 2 #2

        for i in range(X[0].shape[0]):
            for j in range(X[0].shape[1]):
                for k in range(X[0].shape[2]):
                    noise_value = noise.pnoise3(X[0][i, j, k]*scale,
                                                X[1][i, j, k]*scale,
                                                X[2][i, j, k]*scale,
                                                octaves=octaves,
                                                persistence=persistence,
                                                lacunarity=lacunarity)
                    # A scalar perlin noise field is generated, then, the same values are assigned to every velocity component.
                    U[0][i, j, k] = noise_value
                    U[1][i, j, k] = noise_value
                    U[2][i, j, k] = noise_value

    if IC_type == 'abc_flow':
        # ABC flow initialization (Check Rempel (2009) paper)
        amplitude = 1
        forcing_wavenumber = 5 #5 #0.5
        phase_shift = 0 #cp.pi/4

        U[0] = amplitude * cp.sin(forcing_wavenumber*X[2] + phase_shift) + cp.cos(forcing_wavenumber*X[1] + phase_shift)
        U[1] = amplitude * cp.sin(forcing_wavenumber*X[0] + phase_shift) + cp.cos(forcing_wavenumber*X[2] + phase_shift)
        U[2] = amplitude * cp.sin(forcing_wavenumber*X[1] + phase_shift) + cp.cos(forcing_wavenumber*X[0] + phase_shift)

    if IC_type == 'zero':
        # Initiate the velocities with 0 norm
        U[0] = 0.0
        U[1] = 0.0
        U[2] = 0.0

    if IC_type == 'linear':
        # Creates a field for plot testing

        # Create a grid of coordinates (x, y, z)
        x = cp.linspace(0, 1/3, N)  # Grid values between 0 and 1
        y = cp.linspace(0, 1/3, N)
        z = cp.linspace(0, 1/3, N)
        X, Y, Z = cp.meshgrid(x, y, z, indexing='ij')

        # Compute velocity components
        U[0] = X + Y + Z  # Velocity in the x-direction
        U[1] = X + Y + Z  # Velocity in the y-direction
        U[2] = X + Y + Z  # Velocity in the z-direction

    print('pre_spectral')

    #if rank == 0: print('U', 'rank:', rank, U)
    #if rank == 0: print('U_hat', 'rank:', rank, U_hat)
    
    # On spectral space:
    for i in range(3):
        print('inside_loop', i)
        U_hat[i] = fftn_mpi(U[i], U_hat[i])

    # if rank == 0: print('U_hat_post', U_hat.device, 'rank:', rank, U_hat)
    
    return U, U_hat


def custom_alltoall(sendbuf, axis=0, comm=MPI.COMM_WORLD):
    """
    Custom all-to-all exchange using send and recv.#
    Each process's sendbuf is assumed to be a CuPy array.
    The array is divided along the specified axis into chunks,
    and each process sends its chunks to every other process.

    Args:
        sendbuf: A CuPy array of any shape.
        axis: The axis along which to split the array (default: 0).
        comm: MPI communicator (default: MPI.COMM_WORLD).

    Returns:
        recvbuf: A CuPy array of the same shape as sendbuf with data
                 received from all processes.
    """
    # rank = comm.Get_rank()
    # size = comm.Get_size()
    
    # Ensure sendbuf is a CuPy array
    if not isinstance(sendbuf, cp.ndarray):
        raise TypeError("sendbuf must be a CuPy array")
    
    # Get the shape of the sendbuf
    shape = sendbuf.shape
    # N = shape[axis]  # Size of the axis to be split
    
    # Calculate the chunk size along the specified axis
    # chunk_size = N // size
    
    # Split the sendbuf into chunks for each process
    send_chunks = []
    for i in range(size):
        # Create a slice object to dynamically slice along the specified axis
        slice_obj = [slice(None)] * len(shape)
        slice_obj[axis] = slice(i * Np, (i + 1) * Np)
        send_chunks.append(sendbuf[tuple(slice_obj)])
    
    # Create an empty receive buffer
    recvbuf = cp.empty_like(sendbuf)
    
    # Synchronize the GPU before MPI communication
    cp.cuda.Device().synchronize()
    
    # Loop over all peers (ranks) to perform pairwise exchange.
    for peer in range(size):
        if peer == rank:
            # For self communication, copy directly.
            slice_obj = [slice(None)] * len(shape)
            slice_obj[axis] = slice(peer * Np, (peer + 1) * Np)
            recvbuf[tuple(slice_obj)] = send_chunks[peer]
        else:
            # To avoid deadlock, lower rank sends first, higher rank receives first.
            if rank < peer:
                cp.cuda.Device().synchronize()
                comm_nccl.send(send_chunks[peer].data.ptr, send_chunks[peer].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)                    #Remember of the x2!!!
                slice_obj = [slice(None)] * len(shape)
                slice_obj[axis] = slice(peer * Np, (peer + 1) * Np)
                cp.cuda.Device().synchronize()
                comm_nccl.recv(recvbuf[tuple(slice_obj)].data.ptr, recvbuf[tuple(slice_obj)].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)
            else:
                slice_obj = [slice(None)] * len(shape)
                slice_obj[axis] = slice(peer * Np, (peer + 1) * Np)
                cp.cuda.Device().synchronize()
                comm_nccl.recv(recvbuf[tuple(slice_obj)].data.ptr, recvbuf[tuple(slice_obj)].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)
                cp.cuda.Device().synchronize()
                comm_nccl.send(send_chunks[peer].data.ptr, send_chunks[peer].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)
    
    # Synchronize the GPU after MPI communication
    cp.cuda.Device().synchronize()
    
    return recvbuf






def fftn_mpi(u, fu):
    '''
    Perform forward Fourier transform using MPI
    '''
    #print('inside_fftn_mpi')
    #if rank == 0: print('u', 'rank:', rank, u, U.device)
    #if rank == 0: print('fu', 'rank:', rank, fu, U_hat.device)
        
    #if rank == 0: print('Uc_hatT before', 'rank:', rank, Uc_hatT, U_hat.device)    
    Uc_hatT[:] = rfft2(u, axes=(1, 2))
    #if rank == 0: print('Uc_hatT after', 'rank:', rank, Uc_hatT, U_hat.device)
    
    #print('uc_hatT')

    #if rank == 0: print('fu before', 'rank:', rank, fu, U_hat.device)
    fu[:] = cp.rollaxis(Uc_hatT.reshape(Np, size, Np, N//2+1), 1).reshape(fu.shape)
    #if rank == 0: print('fu after', 'rank:', rank, fu, U_hat.device)
    
    #print('fu rollaxis')
    #cp.cuda.Stream.null.synchronize()
    #print('sync')
    
    #comm.Alltoall(MPI.IN_PLACE, [fu, MPI.DOUBLE_COMPLEX])
    #if rank == 0: print('fu before ALL TO ALL', 'rank:', rank, fu, U_hat.device)
    fu = custom_alltoall(fu, comm=comm)
    #if rank == 0: print('fu after ALL TO ALL', 'rank:', rank, fu, U_hat.device)
    
    #print('comm alltoall')
    fu[:] = fft(fu, axis=0)
    #print('fu last fft')
    return fu

def ifftn_mpi(fu, u):
    Uc_hat[:] = ifft(fu, axis=0)
    #comm.Alltoall(MPI.IN_PLACE, [Uc_hat, MPI.DOUBLE_COMPLEX])
    #cp.cuda.Stream.null.synchronize()
    Uc_hat[:] = custom_alltoall(Uc_hat, comm=comm)
    Uc_hatT[:] = cp.rollaxis(Uc_hat.reshape((size, Np, Np, N//2+1)), 1).reshape(Uc_hatT.shape)
    u[:] = irfft2(Uc_hatT, axes=(1, 2))
    return u

def ifftn_serial(fu, u):
    '''
    Perform inverse Fourier transform (serial version)
    '''
    u[:] = np.fft.irfftn(fu)
    return u



def Cross(a, b, c):
    '''
    Compute the cross product of two vectors
    '''
    c[0] = fftn_mpi(a[1]*b[2]-a[2]*b[1], c[0])
    c[1] = fftn_mpi(a[2]*b[0]-a[0]*b[2], c[1])
    c[2] = fftn_mpi(a[0]*b[1]-a[1]*b[0], c[2])
    return c

def Curl(a, c):
    '''
    Compute the curl of a vector field
    '''
    c[2] = ifftn_mpi(1j*(K[0]*a[1]-K[1]*a[0]), c[2])
    c[1] = ifftn_mpi(1j*(K[2]*a[0]-K[0]*a[2]), c[1])
    c[0] = ifftn_mpi(1j*(K[1]*a[2]-K[2]*a[1]), c[0])
    return c

def ComputeRHS(dU, rk):
    '''
    Compute the right-hand side of the Navier-Stokes equations
    '''
    if rk > 0:
        for i in range(3):
            U[i] = ifftn_mpi(U_hat[i], U[i])
    curl[:] = Curl(U_hat, curl)
    dU = Cross(U, curl, dU)
    dU *= dealias
    P_hat[:] = cp.sum(dU*K_over_K2, 0, out=P_hat)
    dU -= P_hat*K
    dU -= viscosity*K2*U_hat
    return dU


# --- MAIN CODE ---

# Initialize MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

print(f'rank: {rank}, size: {size}')

# Assign GPU based on rank
gpu_id = rank % size
cp.cuda.Device(gpu_id).use()
print(f"Rank {rank} is using GPU {gpu_id}")

# Initialize NCCL
if rank == 0:                                                # Generate a unique NCCL ID only on rank 0 and share it
    unique_id = nccl.get_unique_id()
else:
    unique_id = None
unique_id = comm.bcast(unique_id, root=0)                # Broadcast the NCCL unique ID to all ranks
comm_nccl = nccl.NcclCommunicator(size, unique_id, rank) # Create NCCL communicator

plot_filenames = []

# SIMULATION PARAMETERS:
viscosity = 1/1600                  # Viscosity = 1/Reynolds. Suggestion: Reynolds -> 1600
t_f = 20                            # Final physical time
dt = 0.00703125 #1024: 0.0017578125; 512: 0.003515625; 256: 0.00703125 # 128: 0.0140625; 64: 0.028125                           # Time step. Suggestion: 0.05, 0.15 or 0.01
N = 2**8                           # Grid dimension
IC_type = 'taylor_green'            # Initial conditions.
n_steps = int(cp.ceil(t_f/dt))      # Number of frames in the simulation
every = int(cp.ceil(n_steps/10))    # Plot and or calculate "every" iterations:
n_steps = int(cp.ceil(t_f/dt))      # Number of frames in the simulation
Np = N//size                        # Lenght of the slab

# Coordinates and wave numbers:
X = cp.mgrid[rank*Np:(rank+1)*Np, :N, :N].astype(float)*2*cp.pi/N # 2*pi is the lenght of the physical domain
kx = fftfreq(N, 1./N)
kz = kx[:(N//2+1)].copy()
kz[-1] *= -1
K = cp.array(np.meshgrid(kx, kx[rank*Np:(rank+1)*Np], kz, indexing='ij'), dtype=int)
K2 = cp.sum(K*K, 0, dtype=int)
K_over_K2 = K.astype(float)/cp.where(K2 == 0, 1, K2).astype(float)

# Define dealias:
kmax_dealias = 2./3.*(N//2+1)
dealias = cp.array((abs(K[0]) < kmax_dealias)*(abs(K[1]) < kmax_dealias)*
                (abs(K[2]) < kmax_dealias), dtype=bool)

# Preallocate arrays
U = cp.empty((3, Np, N, N))
U_hat = cp.zeros((3, N, Np, N//2+1), dtype=complex) # np.empty((3, N, Np, N//2+1), dtype=complex)
P = cp.empty((Np, N, N))
P_hat = cp.empty((N, Np, N//2+1), dtype=complex)
U_hat0 = cp.empty((3, N, Np, N//2+1), dtype=complex)
U_hat1 = cp.empty((3, N, Np, N//2+1), dtype=complex)
dU = cp.empty((3, N, Np, N//2+1), dtype=complex)
Uc_hat = cp.empty((N, Np, N//2+1), dtype=complex)
Uc_hatT = cp.empty((Np, N, N//2+1), dtype=complex) # np.empty((Np, N, N//2+1), dtype=complex)
curl = cp.empty((3, Np, N, N))

# Runge-Kutta coefficients:
a = [1./6., 1./3., 1./3., 1./6.]
b = [0.5, 0.5, 1.]

# --- MAIN LOOP: ---
pbar = tqdm(total=int(n_steps))

U, U_hat = IC_3D(X, IC_type)

for n in range(min(100, n_steps + 1)): # n_steps + 1 

    #Initialize U_hat1 and U_hat0 (copies of U_hat used on intermediate steps of Runge-Kutta)
    U_hat1[:] = U_hat0[:] = U_hat

    #Runge-Kutta integration
    for rk in range(4):
        
        dU = ComputeRHS(dU, rk)                #Compute the right-hand side of N-S equations
        
        if rk < 3:
            U_hat[:] = U_hat0 + b[rk]*dt*dU    # Update U_hat based on R-K coefficients b
            
        U_hat1[:] += a[rk]*dt*dU               # Update U_hat1 based on R-K coefficients a

    U_hat[:] = U_hat1[:]                       # Update U-hat with the final results on U_hat1
    
    pbar.update(1)

pbar.close()


Overwriting /kaggle/working/test_mpi.py


## Run:

In [1]:
!mpiexec --allow-run-as-root -np 2 python test_mpi.py

/usr/bin/sh: 1: mpiexec: not found


# Multi-CPU version:

In [11]:
%%writefile /kaggle/working/test_mpi.py

import time
import noise
import numpy as np
from tqdm import tqdm
from mpi4py import MPI
from numpy.fft import fftfreq, fft, ifft, irfft2, rfft2, fftshift, ifftshift, fftn, irfftn

def IC_3D(X, IC_type):
    '''
    This function initializes the velocity field in Fourier space based on the initial condition type
    '''
    if IC_type == 'random_vel':
        # Random velocity initial condition (not a very good IC for 3D turbulence)
        U[0] = np.random.rand(*X[0].shape)
        U[1] = np.random.rand(*X[0].shape)
        U[2] = np.random.rand(*X[0].shape)

        #Resize:
        U[0] /= np.max(U[0])
        U[1] /= np.max(U[1])
        U[2] /= np.max(U[2])

    if IC_type == 'taylor_green':
        # Taylor-Green vortex initial conditions (Check Mortensen (2016) paper)
        U[0] = np.sin(X[0])*np.cos(X[1])*np.cos(X[2])
        U[1] = -np.cos(X[0])*np.sin(X[1])*np.cos(X[2])
        U[2] = 0

    if IC_type == 'taylor_green_noise':
        # Taylor-Green vortex with added noise initial condition
        U[0] = np.sin(X[0])*np.cos(X[1])*np.cos(X[2])
        U[1] = -np.cos(X[0])*np.sin(X[1])*np.cos(X[2])
        U[2] = 0

        #Add white noise:
        epsilon = 0.1
        U[0] += epsilon*np.random.rand(*U[0].shape)
        U[1] += epsilon*np.random.rand(*U[1].shape)
        U[2] += epsilon*np.random.rand(*U[2].shape)

    if IC_type == 'perlin_noise':
        # Perlin noise CURL initial condition

        scale = 1/4 #0.25
        octaves = 5 #2
        persistence = 0.4 #0.5
        lacunarity = 2 #2

        for i in range(X[0].shape[0]):
            for j in range(X[0].shape[1]):
                for k in range(X[0].shape[2]):
                    noise_value = noise.pnoise3(X[0][i, j, k]*scale,
                                                X[1][i, j, k]*scale,
                                                X[2][i, j, k]*scale,
                                                octaves=octaves,
                                                persistence=persistence,
                                                lacunarity=lacunarity)
                    # A scalar perlin noise field is generated, then, the same values are assigned to every velocity component.
                    U[0][i, j, k] = noise_value
                    U[1][i, j, k] = noise_value
                    U[2][i, j, k] = noise_value

    if IC_type == 'abc_flow':
        # ABC flow initialization (Check Rempel (2009) paper)
        amplitude = 1
        forcing_wavenumber = 5 #5 #0.5
        phase_shift = 0 #cp.pi/4

        U[0] = amplitude * np.sin(forcing_wavenumber*X[2] + phase_shift) + np.cos(forcing_wavenumber*X[1] + phase_shift)
        U[1] = amplitude * np.sin(forcing_wavenumber*X[0] + phase_shift) + np.cos(forcing_wavenumber*X[2] + phase_shift)
        U[2] = amplitude * np.sin(forcing_wavenumber*X[1] + phase_shift) + np.cos(forcing_wavenumber*X[0] + phase_shift)

    if IC_type == 'zero':
        # Initiate the velocities with 0 norm
        U[0] = 0.0
        U[1] = 0.0
        U[2] = 0.0

    if IC_type == 'linear':
        # Creates a field for plot testing

        # Create a grid of coordinates (x, y, z)
        x = cp.linspace(0, 1/3, N)  # Grid values between 0 and 1
        y = cp.linspace(0, 1/3, N)
        z = cp.linspace(0, 1/3, N)
        X, Y, Z = cp.meshgrid(x, y, z, indexing='ij')

        # Compute velocity components
        U[0] = X + Y + Z  # Velocity in the x-direction
        U[1] = X + Y + Z  # Velocity in the y-direction
        U[2] = X + Y + Z  # Velocity in the z-direction
        
    #if rank == 0: print('U', 'rank:', rank, U)
    #if rank == 0: print('U_hat', 'rank:', rank, U_hat)
    
    # On spectral space:
    for i in range(3):
        print('inside_loop', i)
        U_hat[i] = fftn_mpi(U[i], U_hat[i])

    # if rank == 0: print('U_hat_post', 'rank:', rank, U_hat)
    
    return U, U_hat

def fftn_mpi(u, fu):
    '''
    Perform forward Fourier transform using MPI
    '''
    #print('inside_fftn_mpi')
    #if rank == 0: print('u', 'rank:', rank, u)
    #if rank == 0: print('fu', 'rank:', rank, fu)

    #if rank == 0: print('Uc_hatT before', 'rank:', rank, Uc_hatT)    
    Uc_hatT[:] = rfft2(u, axes=(1, 2))
    #if rank == 0: print('Uc_hatT after', 'rank:', rank, Uc_hatT)    

    #if rank == 0: print('fu before', 'rank:', rank, fu)
    fu[:] = np.rollaxis(Uc_hatT.reshape(Np, size, Np, N//2+1), 1).reshape(fu.shape)
    #if rank == 0: print('fu after', 'rank:', rank, fu)

    #if rank == 0: print('fu before ALL TO ALL', 'rank:', rank, fu)
    comm.Alltoall(MPI.IN_PLACE, [fu, MPI.DOUBLE_COMPLEX])
    #if rank == 0: print('fu after ALL TO ALL', 'rank:', rank, fu)
    
    fu[:] = fft(fu, axis=0)
    return fu

def ifftn_mpi(fu, u):
    '''
    Perform inverse Fourier transform using MPI
    '''
    Uc_hat[:] = ifft(fu, axis=0)
    comm.Alltoall(MPI.IN_PLACE, [Uc_hat, MPI.DOUBLE_COMPLEX])
    Uc_hatT[:] = np.rollaxis(Uc_hat.reshape((size, Np, Np, N//2+1)), 1).reshape(Uc_hatT.shape)
    u[:] = irfft2(Uc_hatT, axes=(1, 2))
    return u

def ifftn_serial(fu, u):
    '''
    Perform inverse Fourier transform (serial version)
    '''
    #Uc_hat[:] = ifft(fu, axis=0)
    #comm.Alltoall(MPI.IN_PLACE, [Uc_hat, MPI.DOUBLE_COMPLEX])
    #Uc_hatT[:] = np.rollaxis(Uc_hat.reshape((num_processes, N, Np, N//2+1)), 1).reshape(Uc_hatT.shape)
    #u[:] = irfft2(Uc_hatT, axes=(1, 2))
    u[:] = irfftn(fu)
    return u

def Cross(a, b, c):
    '''
    Compute the cross product of two vectors
    '''
    c[0] = fftn_mpi(a[1]*b[2]-a[2]*b[1], c[0])
    c[1] = fftn_mpi(a[2]*b[0]-a[0]*b[2], c[1])
    c[2] = fftn_mpi(a[0]*b[1]-a[1]*b[0], c[2])
    return c

def Curl(a, c):
    '''
    Compute the curl of a vector field
    '''
    c[2] = ifftn_mpi(1j*(K[0]*a[1]-K[1]*a[0]), c[2])
    c[1] = ifftn_mpi(1j*(K[2]*a[0]-K[0]*a[2]), c[1])
    c[0] = ifftn_mpi(1j*(K[1]*a[2]-K[2]*a[1]), c[0])
    return c

def ComputeRHS(dU, rk):
    '''
    Compute the right-hand side of the Navier-Stokes equations
    '''
    if rk > 0:
        for i in range(3):
            U[i] = ifftn_mpi(U_hat[i], U[i])
    curl[:] = Curl(U_hat, curl)
    dU = Cross(U, curl, dU)
    dU *= dealias
    P_hat[:] = np.sum(dU*K_over_K2, 0, out=P_hat)
    dU -= P_hat*K
    dU -= viscosity*K2*U_hat
    return dU



#   --- // ---



#List to save plots:
plot_filenames = []

# MPI SETUP:
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
print('num_processes', size, 'rank', rank)

# SIMULATION PARAMETERS:
viscosity = 1/1600                  # Viscosity = 1/Reynolds. Suggestion: Reynolds -> 1600
t_f = 20                            # Final physical time
dt = 0.00703125 #1024: 0.0017578125; 512: 0.003515625; 256: 0.00703125 # 128: 0.0140625; 64: 0.028125                           # Time step. Suggestion: 0.05, 0.15 or 0.01
N = 2**8                           # Grid dimension
IC_type = 'taylor_green'            # Initial conditions.
n_steps = int(np.ceil(t_f/dt))      # Number of frames in the simulation
every = int(np.ceil(n_steps/10))    # Plot and or calculate "every" iterations:
n_steps = int(np.ceil(t_f/dt))      # Number of frames in the simulation
Np = N//size                        # Lenght of the slab


n_steps = int(np.ceil(t_f/dt)) #Number of frames in the simulation

# Coordinates and wave numbers:
X = np.mgrid[rank*Np:(rank+1)*Np, :N, :N].astype(float)*2*np.pi/N # 2*pi is the lenght of the physical domain
kx = fftfreq(N, 1./N)
kz = kx[:(N//2+1)].copy()
kz[-1] *= -1
K = np.array(np.meshgrid(kx, kx[rank*Np:(rank+1)*Np], kz, indexing='ij'), dtype=int)
K2 = np.sum(K*K, 0, dtype=int)
K_over_K2 = K.astype(float)/np.where(K2 == 0, 1, K2).astype(float)

# Define dealias:
kmax_dealias = 2./3.*(N//2+1)
dealias = np.array((abs(K[0]) < kmax_dealias)*(abs(K[1]) < kmax_dealias)*
                (abs(K[2]) < kmax_dealias), dtype=bool)

# Preallocate arrays
U = np.empty((3, Np, N, N))
U_hat = np.zeros((3, N, Np, N//2+1), dtype=complex) # np.empty((3, N, Np, N//2+1), dtype=complex)
P = np.empty((Np, N, N))
P_hat = np.empty((N, Np, N//2+1), dtype=complex)
U_hat0 = np.empty((3, N, Np, N//2+1), dtype=complex)
U_hat1 = np.empty((3, N, Np, N//2+1), dtype=complex)
dU = np.empty((3, N, Np, N//2+1), dtype=complex)
Uc_hat = np.empty((N, Np, N//2+1), dtype=complex)
Uc_hatT = np.zeros((Np, N, N//2+1), dtype=complex) # np.empty((Np, N, N//2+1), dtype=complex)
curl = np.empty((3, Np, N, N))

# Runge-Kutta coefficients:
a = [1./6., 1./3., 1./3., 1./6.]
b = [0.5, 0.5, 1.]

# --- MAIN LOOP: ---
pbar = tqdm(total=int(n_steps))

# Initial velocity initial field (in physical and spectral space):
U, U_hat = IC_3D(X, IC_type)

for n in range(min(100, n_steps + 1)): # n_steps + 1 

    #Initialize U_hat1 and U_hat0 (copies of U_hat used on intermediate steps of Runge-Kutta)
    U_hat1[:] = U_hat0[:] = U_hat

    #Runge-Kutta integration
    for rk in range(4):
        
        dU = ComputeRHS(dU, rk)                #Compute the right-hand side of N-S equations
        
        if rk < 3:
            U_hat[:] = U_hat0 + b[rk]*dt*dU    # Update U_hat based on R-K coefficients b
            
        U_hat1[:] += a[rk]*dt*dU               # Update U_hat1 based on R-K coefficients a

    U_hat[:] = U_hat1[:]                       # Update U-hat with the final results on U_hat1

    pbar.update(1)

pbar.close()


Overwriting /kaggle/working/test_mpi.py


## Run:

In [18]:
!mpiexec --allow-run-as-root -np 16 python test_mpi.py 

num_processes 16 rank 8
num_processes 16 rank 12
num_processes 16 rank 14
num_processes 16 rank 0
num_processes 16 rank 2
num_processes 16 rank 4
num_processes 16 rank 9
num_processes 16 rank 10
num_processes 16 rank 11
num_processes 16 rank 13
num_processes 16 rank 15
num_processes 16 rank 1
num_processes 16 rank 3
num_processes 16 rank 6
num_processes 16 rank 7
num_processes 16 rank 5
  0%|          | 0/2845 [00:00<?, ?it/s]inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 0
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 1
inside_loop 2
inside_loop 2
inside_loop 2
inside_loop 2
inside_loop 2
inside_loop 2
inside_loop 2
inside_loop 2
inside_loo

# Delete all files on the directory:

In [23]:
!rm -rf /kaggle/content/*